# Chunking Tests and Trials

## Table of Contents

- [ 1 - Introduction](#1)
  - [ 1.1 Importing necessary libraries](#1-1)
  - [ 1.2 Importing the data](#1-2)
  - [ 1.3 Clean the data](#1-3)
  - [ 1.4 Create page chunks](#1-4)
- [ 2 - Fixed-size chunking](#2)
  - [ 2.1 Example Chunking Code](#2-1)
  - [ 2.2 Chunking with overlap](#2-2)
- [ 3 - Variable-size chunking - Recursive Character Splitting](#3)
  - [ 3.1 Variable-size chunking methods](#3-1)
  - [ 3.2 Mixing fixed and variable-sized chunking](#3-2)
- [ 4 - Semantic chunking ](#4)
- [ 5 - LLM Chunking](#5)
- [ 6 - Searching ](#6)
- [ 7 - Evaluation of chunking methods and metrics](#7)

<a id='1'></a>
## Introduction

<a id='1-1'></a>
### 1.1 - Importing necessary libraries

In [125]:
import re
import json
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, Dict, Any, List, Tuple, Optional
from collections import Counter

import pymupdf  # PyMuPDF
# Data / vector math
import numpy as np

# Embeddings (for semantic chunking + search)
from sentence_transformers import SentenceTransformer
import math

#### Configuration variables

In [46]:
DOCS_ROOT = Path("docs")
TOPIC_FOLDERS = {"general", "mama"}  # extend later: {"general","mama","prostata",...}
PAGES_JSONL = Path("docs/pages.jsonl")
CHUNKS_JSONL = Path("docs/chunks.jsonl")

# Optional: only ingest Spanish pages
ONLY_LANG = "es"   # set to None to ingest all languages

#Document metadata structure
@dataclass
class DocMeta:
    doc_id: str #stable unique ID
    topic: str #mama, general, prostata, etc.
    lang: str #language
    source: str #novartis, gepac, etc.
    slug: str #descriptive text
    version: str #version (v1, v2) or year of publication
    path: str #where the file is on disk
    file_hash: str #unique hash to detect changes

<a id='1-3'></a>
### 1.3 - Cleaning the data

#### Remove headings and footers

In [53]:
def extract_header_footer_candidates(text: str, *, top_n: int = 3, bottom_n: int = 3) -> list[str]:
    """
    Take the first and last N non-empty lines of a page as header/footer candidates.
    """
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return []
    return lines[:top_n] + lines[-bottom_n:]


In [54]:
def detect_repeated_headers_footers(
    page_texts: list[str],
    *,
    min_freq_ratio: float = 0.5,
    top_n: int = 3,
    bottom_n: int = 3,
) -> set[str]:
    """
    Identify lines that appear on many pages → likely headers/footers.
    """
    counter = Counter()
    n_pages = len(page_texts)

    for txt in page_texts:
        candidates = extract_header_footer_candidates(txt, top_n=top_n, bottom_n=bottom_n)
        for c in candidates:
            counter[c] += 1

    # Keep lines that appear in at least X% of pages
    repeated = {
        line
        for line, count in counter.items()
        if count / n_pages >= min_freq_ratio
    }

    return repeated


#### Text Cleanup helpers

In [58]:
def normalize_pdf_text(text: str,*,headers_footers: set[str] | None = None,) -> str:
    """
    Clean PDF text and remove detected headers/footers.
    """
    if not text:
        return ""

    # Split into lines first (important for header/footer removal)
    lines = [l.rstrip() for l in text.splitlines()]

    if headers_footers:
        lines = [
            l for l in lines
            if l.strip() and l.strip() not in headers_footers
        ]

    text = "\n".join(lines)

    # Fix hyphenation across line breaks
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Merge single newlines into spaces (keep paragraph breaks)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()



In [13]:
def _normalize_stem(stem: str) -> str:
    # Convert common unicode dashes to ASCII hyphen
    stem = stem.replace("–", "-").replace("—", "-").replace("−", "-")
    # Normalize whitespace (just in case)
    stem = stem.strip()
    # Collapse multiple underscores
    stem = re.sub(r"_+", "_", stem)
    return stem

<a id='1-2'></a>
### 1.2 - Importing the data

#### Hash helpers (for traceability)

In [14]:
def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="ignore")).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


#### Filename metadata parser

In [15]:
# The name convention for the metadata is:
# <topic>_<lang>_<source>_<slug>_<version>.<ext>
# Example: mama_es_gepac_guia-de-practica-clinica_v1.pdf
# Example: prostata_es_novartis_linea-de-tiempo_2023.pdf

def parse_filename_meta(pdf_path: Path) -> Dict[str, str]:
    """
    Parse metadata from filename.
    """
    stem_raw = pdf_path.stem
    stem = _normalize_stem(stem_raw)
    parts = stem.split("_", 4)
    if len(parts) != 5:
        raise ValueError(f"Bad filename format: {pdf_path.name}")

    topic, lang, source, slug, version = parts

    return {
        "doc_id": stem.lower(),
        "topic": topic.lower(),
        "lang": lang.lower(),
        "source": source.lower(),
        "slug": slug,
        "version": version.lower(),
        "path": str(pdf_path.as_posix()),
        "file_hash": sha256_file(pdf_path),
    }


<a id='1-4'></a>
### 1.4 - Create page chunks

In [ ]:
#Final extract_page_chunk after debugging (v3)
def extract_page_chunks(pdf_path: Path) -> List[Dict[str, Any]]:
    meta = parse_filename_meta(pdf_path)
    if ONLY_LANG is not None and meta["lang"] != ONLY_LANG:
        print(f"[ERR] {pdf_path.name} meta.lang = {meta["lang"]} ONLY_LANG = {ONLY_LANG}")
        return []

    with pymupdf.open(pdf_path) as doc:
        raw_texts = [doc.load_page(p).get_text("text") for p in range(doc.page_count)]
        print(f"    [INFO] {pdf_path.name} pages = {doc.page_count}")

    headers_footers = detect_repeated_headers_footers(raw_texts)

    records: List[Dict[str, Any]] = []
    kept, skipped = 0, 0

    for p, raw in enumerate(raw_texts):
        page_num = p + 1
        text = normalize_pdf_text(raw, headers_footers=headers_footers)

        if len(text) < 50:
            skipped += 1
            # debug útil: muestra longitud raw vs cleaned
            #if len(raw.strip()) > 0:
            #    print(f"[SKIP] {pdf_path.name} p{page_num:03d}: cleaned={len(text)} raw={len(raw.strip())}")
            continue

        kept += 1
        page_id = f'{meta["doc_id"]}__p{page_num:03d}'
        records.append({
            "page_id": page_id,
            "text": text,
            "page": page_num,
            **meta,
            "text_hash": sha256_text(text),
            "char_len": len(text),
        })

    #print(f"[INFO] {pdf_path.name}: kept={kept}, skipped={skipped}, total_pages={len(raw_texts)}")
    #print("meta.lang =", meta["lang"], "ONLY_LANG =", ONLY_LANG)
    #print("Detected header/footer lines:", len(headers_footers))
    #for x in list(sorted(headers_footers))[:20]:
    #    print("-", x)

    return records


In [56]:
def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
#Find all pdfs of specified TOPIC_FOLDERS
def iter_pdf_paths(root: Path) -> Iterator[Path]:
    # Case 1: use all PDFs under root
    if TOPIC_FOLDERS == "All":
        yield from root.rglob("*.pdf")
        return
 
    # Case 2: TOPIC_FOLDERS is an iterable of folder names
    for topic in TOPIC_FOLDERS:
        folder = root / topic
        if folder.exists():
            yield from folder.rglob("*.pdf")

Test

In [103]:
#Test iter_pdf_paths function for case 1 and 2
TOPIC_FOLDERS_TEMP = TOPIC_FOLDERS
TOPIC_FOLDERS = {"general", "mama"}
print(f"Topic folders: {TOPIC_FOLDERS}")
pdf_roots = list(iter_pdf_paths(DOCS_ROOT))
if not pdf_roots:
        raise FileNotFoundError(f"No PDFs found under {DOCS_ROOT}/{TOPIC_FOLDERS}")
print("Found PDFs:", len(pdf_roots))
for i in range(len(pdf_roots)): 
    print(f"[{i+1}] {pdf_roots[i]}")

TOPIC_FOLDERS = "All"
print(f"Topic folders: {TOPIC_FOLDERS}")
pdf_roots = list(iter_pdf_paths(DOCS_ROOT))
if not pdf_roots:
        raise FileNotFoundError(f"No PDFs found under {DOCS_ROOT}/{TOPIC_FOLDERS}")
print("Found PDFs:", len(pdf_roots))
for i in range(len(pdf_roots)): 
    print(f"[{i+1}] {pdf_roots[i]}")
TOPIC_FOLDERS = TOPIC_FOLDERS_TEMP #Restore original TOPIC_FOLDERS after test


Topic folders: {'mama', 'general'}
Found PDFs: 6
[1] docs\mama\mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf
[2] docs\mama\mama_es_esmo_guia-para-pacientes_v1.pdf
[3] docs\mama\mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf
[4] docs\mama\mama_es_novartis_guia-pacientes-CM_2025.pdf
[5] docs\general\general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf
[6] docs\general\general_es_pfizer_manual-pacientes_2007.pdf
Topic folders: All
Found PDFs: 10
[1] docs\general\general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf
[2] docs\general\general_es_pfizer_manual-pacientes_2007.pdf
[3] docs\mama\mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf
[4] docs\mama\mama_es_esmo_guia-para-pacientes_v1.pdf
[5] docs\mama\mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf
[6] docs\mama\mama_es_novartis_guia-pacientes-CM_2025.pdf
[7] docs\prostata\ES____Cáncer_de_Próstata__Guía_para_Pacientes.pdf
[8] docs\prostata\guia-pacientes-cancer-prostata.pdf
[9] docs\prostata\GUÍA_CÁNCER_DE_PRÓSTAT

Run extraction for all PDFs and save pages.jsonl

In [106]:
pdf_roots = list(iter_pdf_paths(DOCS_ROOT))
print("Found PDFs:", len(pdf_roots))
for i in range(len(pdf_roots)): 
    print(f"[{i+1}] {pdf_roots[i]}")

all_pages: List[Dict[str, Any]] = []

for p in pdf_roots:
    try:
        pages = extract_page_chunks(p)
        all_pages.extend(pages)
        print(f"[OK] {p.name} → {len(pages)} pages")
    except Exception as e:
        print(f"[ERR] {p.name}: {e}")

print("\nTotal page chunks:", len(all_pages))

write_jsonl(PAGES_JSONL, all_pages)
print("Saved to:", PAGES_JSONL.resolve())


Found PDFs: 6
[1] docs\mama\mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf
[2] docs\mama\mama_es_esmo_guia-para-pacientes_v1.pdf
[3] docs\mama\mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf
[4] docs\mama\mama_es_novartis_guia-pacientes-CM_2025.pdf
[5] docs\general\general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf
[6] docs\general\general_es_pfizer_manual-pacientes_2007.pdf
[ERR] mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf meta.lang = en ONLY_LANG = es
[OK] mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf → 0 pages
    [INFO] mama_es_esmo_guia-para-pacientes_v1.pdf pages = 76
[OK] mama_es_esmo_guia-para-pacientes_v1.pdf → 76 pages
    [INFO] mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf pages = 122
[OK] mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf → 118 pages
    [INFO] mama_es_novartis_guia-pacientes-CM_2025.pdf pages = 126
[OK] mama_es_novartis_guia-pacientes-CM_2025.pdf → 123 pages
    [INFO] general_es_gepac_guia-toxicidad-quimioterapia_

Sanity check

In [107]:
# Inspect first 2 pages
with PAGES_JSONL.open("r", encoding="utf-8") as f:
    for _ in range(2):
        obj = json.loads(f.readline())
        print(obj["page_id"], "| page:", obj["page"], "| source:", obj["source"])
        print(obj["text"][:300], "\n")


mama_es_esmo_guia-para-pacientes_v1__p001 | page: 1 | source: esmo
esmo.org Serie de guías ESMO para pacientes basada en la guía de práctica clínica de la ESMO ¿Qué es el cáncer de mama? Déjenos responder a algunas de sus preguntas Cáncer de mama 

mama_es_esmo_guia-para-pacientes_v1__p002 | page: 2 | source: esmo
2 Una guía ESMO para pacientes Información para el paciente basada en las guías de práctica clínica de la ESMO Esta guía ha sido preparada para ayudarle a usted, así como a sus amigos, familiares y cuidadores, a comprender mejor el cáncer de mama y su tratamiento. Incluye información sobre el cáncer 



### Debug

ERROR POR RESOLVER: Hay un documento que la función extract_pdf_chunks() no consigue leer bien y devuelve 0 páginas, pero sin lo nuevo de extract heading y footer sí que se lee bien

In [65]:
def extract_page_chunks_old(pdf_path: Path) -> List[Dict[str, Any]]:
    """
    Extract one record per page from a PDF.
    """
    meta = parse_filename_meta(pdf_path)

    if ONLY_LANG is not None and meta["lang"] != ONLY_LANG:
        return []

    file_hash = sha256_file(pdf_path)
    records = []

    with pymupdf.open(pdf_path) as doc:
        for i in range(doc.page_count):
            page = doc.load_page(i)
            text = normalize_pdf_text(page.get_text("text"))

            if len(text) < 50:   # skip empty/noisy pages
                continue

            page_num = i + 1
            page_id = f'{meta["doc_id"]}__p{page_num:03d}'

            records.append({
                "page_id": page_id,
                "text": text,
                "page": page_num,

                # document metadata
                **meta,

                # traceability
                "file_hash": file_hash,
                "text_hash": sha256_text(text),
                "char_len": len(text),
            })

    return records


In [72]:
doc = Path("docs/general/general_es_pfizer_manual-pacientes_2007.pdf")
p_doc = extract_page_chunks_old(doc)
print(f"{doc.name} --> pages {len(p_doc)}")
print(p_doc)

p_doc_v2 = extract_page_chunks(doc)
print(f"{doc.name} --> pages {len(p_doc_v2)}")
print(p_doc_v2)

general_es_pfizer_manual-pacientes_2007.pdf --> pages 182
[{'page_id': 'general_es_pfizer_manual-pacientes_2007__p001', 'text': 'EL MANUAL PARA EL PACIENTE ONCOLÓGICO Y SU FAMILIA es fruto de una intensa reflexión y de una gran empatía con el paciente oncológico. Sin intentar rebajar un ápice la importancia que tiene la enfermedad ni negar su dureza, los autores han querido obviar la visión catastrofista, tan habitual cuando se habla del cáncer y han preferido incidir en los aspectos que más pueden servir a los pacientes para que afronten con éxito su situación, hacerles la vida más agradable durante el tratamiento y ayudarles en su retorno a la vida diaria una vez curados. Eminentemente práctico, el libro ofrece información rigurosa a la vez que útil, transmitiendo a la par, un mensaje de ánimo y de confianza. En definitiva, ofrece ideas prácticas y positivas para mejorar la calidad de vida del enfermo oncológico, de sus familiares y de los cuidadores que le atienden. Mª Luisa de Cáce

In [67]:
pdf_path = Path("docs/general/general_es_pfizer_manual-pacientes_2007.pdf")

try:
    doc = pymupdf.open(pdf_path)
    print("Opened OK. Pages:", doc.page_count)
    doc.close()
except Exception as e:
    print("FAILED to open:", e)


Opened OK. Pages: 201


In [68]:
def debug_pdf_extraction(pdf_path: Path, top_n=3, bottom_n=3, min_freq_ratio=0.8):
    with pymupdf.open(pdf_path) as doc:
        raw_texts = [doc.load_page(i).get_text("text") for i in range(doc.page_count)]

    # Conteo de candidatos
    def candidates(text):
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        return lines[:top_n] + lines[-bottom_n:] if lines else []

    counter = Counter()
    for t in raw_texts:
        for c in candidates(t):
            counter[c] += 1

    n_pages = len(raw_texts)
    headers_footers = {line for line, cnt in counter.items() if cnt / n_pages >= min_freq_ratio}

    print("Pages:", n_pages)
    print("Detected header/footer lines:", len(headers_footers))
    for h in list(sorted(headers_footers))[:15]:
        print(" -", h)

    # Compara longitudes por página
    for i, raw in enumerate(raw_texts):
        before = len(raw.strip())
        after = len(normalize_pdf_text(raw, headers_footers=headers_footers))

        print(f"p{i+1:03d} before={before:6d} after={after:6d}")

        # Si una página queda casi vacía, imprime ejemplo
        if before > 200 and after < 80:
            print("\n--- RAW SAMPLE ---")
            print(raw[:800])
            print("\n--- CLEAN SAMPLE ---")
            print(normalize_pdf_text(raw, headers_footers=headers_footers)[:800])
            print("\n==================\n")
            break

# Ejecuta con tu PDF problemático
debug_pdf_extraction(pdf_path)


Pages: 201
Detected header/footer lines: 0
p001 before=  2085 after=  2066
p002 before=   837 after=   830
p003 before=   273 after=   273
p004 before=   403 after=   396
p005 before=     0 after=     0
p006 before=  1991 after=  1973
p007 before=  1442 after=  1432
p008 before=  2753 after=  2746
p009 before=  3014 after=  3007
p010 before=  2378 after=  2375
p011 before=     0 after=     0
p012 before=    69 after=    69
p013 before=     0 after=     0
p014 before=  1528 after=  1506
p015 before=  2337 after=  2304
p016 before=  2830 after=  2810
p017 before=  2503 after=  2480
p018 before=  2324 after=  2284
p019 before=  2893 after=  2878
p020 before=  2636 after=  2606
p021 before=  2841 after=  2815
p022 before=  2343 after=  2315
p023 before=  2678 after=  2659
p024 before=  2749 after=  2720
p025 before=  2944 after=  2909
p026 before=  2477 after=  2461
p027 before=  2300 after=  2276
p028 before=  1295 after=  1280
p029 before=     0 after=     0
p030 before=    63 after=    

In [69]:
kept = 0
skipped = 0
skipped_pages = []

with pymupdf.open(pdf_path) as doc:
    for i in range(doc.page_count):
        text = normalize_pdf_text(doc.load_page(i).get_text("text"), headers_footers=None)
        if len(text) < 50:  # tu umbral real
            skipped += 1
            skipped_pages.append((i+1, len(text)))
            continue
        kept += 1

print("kept:", kept, "skipped:", skipped)
print("first skipped:", skipped_pages[:20])


kept: 182 skipped: 19
first skipped: [(5, 0), (11, 0), (13, 0), (29, 0), (31, 0), (49, 0), (51, 0), (85, 0), (87, 0), (95, 0), (97, 0), (165, 0), (177, 0), (178, 34), (179, 0), (194, 12), (195, 0), (200, 0), (201, 0)]


In [ ]:
def extract_page_chunks_v3(pdf_path: Path) -> List[Dict[str, Any]]:
    meta = parse_filename_meta(pdf_path)
    if ONLY_LANG is not None and meta["lang"] != ONLY_LANG:
        print(f"[ERR] {pdf_path.name} meta.lang = {meta["lang"]} ONLY_LANG = {ONLY_LANG}")
        return []

    with pymupdf.open(pdf_path) as doc:
        raw_texts = [doc.load_page(p).get_text("text") for p in range(doc.page_count)]
        print(f"{pdf_path.name} pages = {doc.page_count}")

    headers_footers = detect_repeated_headers_footers(raw_texts)

    records: List[Dict[str, Any]] = []
    kept, skipped = 0, 0

    for p, raw in enumerate(raw_texts):
        page_num = p + 1
        text = normalize_pdf_text(raw, headers_footers=headers_footers)

        if len(text) < 50:
            skipped += 1
            # debug útil: muestra longitud raw vs cleaned
            if len(raw.strip()) > 0:
                print(f"[SKIP] {pdf_path.name} p{page_num:03d}: cleaned={len(text)} raw={len(raw.strip())}")
            continue

        kept += 1
        page_id = f'{meta["doc_id"]}__p{page_num:03d}'
        records.append({
            "page_id": page_id,
            "text": text,
            "page": page_num,
            **meta,
            "text_hash": sha256_text(text),
            "char_len": len(text),
        })

    print(f"[INFO] {pdf_path.name}: kept={kept}, skipped={skipped}, total_pages={len(raw_texts)}")
    print("meta.lang =", meta["lang"], "ONLY_LANG =", ONLY_LANG)
    print("Detected header/footer lines:", len(headers_footers))
    for x in list(sorted(headers_footers))[:20]:
        print("-", x)

    return records


In [95]:
pdf_path = Path("docs/mama/mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf")
pages = extract_page_chunks_v3(pdf_path)
print("Returned pages:", len(pages))
print("First page_id:", pages[0]["page_id"] if pages else None)


[ERR] mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf meta.lang = en ONLY_LANG = es
Returned pages: 0
First page_id: None


In [83]:
print(pages[0])
print(pages[1])
print(pages[181])

{'page_id': 'general_es_pfizer_manual-pacientes_2007__p001', 'text': 'EL MANUAL PARA EL PACIENTE ONCOLÓGICO Y SU FAMILIA es fruto de una intensa reflexión y de una gran empatía con el paciente oncológico. Sin intentar rebajar un ápice la importancia que tiene la enfermedad ni negar su dureza, los autores han querido obviar la visión catastrofista, tan habitual cuando se habla del cáncer y han preferido incidir en los aspectos que más pueden servir a los pacientes para que afronten con éxito su situación, hacerles la vida más agradable durante el tratamiento y ayudarles en su retorno a la vida diaria una vez curados. Eminentemente práctico, el libro ofrece información rigurosa a la vez que útil, transmitiendo a la par, un mensaje de ánimo y de confianza. En definitiva, ofrece ideas prácticas y positivas para mejorar la calidad de vida del enfermo oncológico, de sus familiares y de los cuidadores que le atienden. Mª Luisa de Cáceres Zurita, de quién surgió la idea original, es Doctora en

In [97]:
pdf_roots = list(iter_pdf_paths(DOCS_ROOT))
print("Found PDFs:", len(pdf_roots))
for i in range(len(pdf_roots)): 
    print(f"[{i+1}] {pdf_roots[i]}")

all_pages: List[Dict[str, Any]] = []

for p in pdf_roots:
    try:
        pages = extract_page_chunks_v3(p)
        all_pages.extend(pages)
        print(f"[OK] {p.name} → {len(pages)} pages")
    except Exception as e:
        print(f"[ERR] {p.name}: {e}")

print("\nTotal page chunks:", len(all_pages))

write_jsonl(PAGES_JSONL, all_pages)
print("Saved to:", PAGES_JSONL.resolve())

Found PDFs: 6
[1] docs\mama\mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf
[2] docs\mama\mama_es_esmo_guia-para-pacientes_v1.pdf
[3] docs\mama\mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf
[4] docs\mama\mama_es_novartis_guia-pacientes-CM_2025.pdf
[5] docs\general\general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf
[6] docs\general\general_es_pfizer_manual-pacientes_2007.pdf
[ERR] mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf meta.lang = en ONLY_LANG = es
[OK] mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf → 0 pages
mama_es_esmo_guia-para-pacientes_v1.pdf pages = 76
[INFO] mama_es_esmo_guia-para-pacientes_v1.pdf: kept=76, skipped=0, total_pages=76
meta.lang = es ONLY_LANG = es
Detected header/footer lines: 1
- Cáncer de mama
[OK] mama_es_esmo_guia-para-pacientes_v1.pdf → 76 pages
mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf pages = 122
[SKIP] mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf p003: cleaned=1 raw=1
[SKIP] mama_es_HUReinaSofia_proto

<a id='2'></a>
## Fixed-size chunking

Import the pages if not loaded already

We are starting to chunk from the pages because they are a stable "citation unit" and a convenient "raw text unit".

In [ ]:
PAGES_JSONL = Path("docs/pages.jsonl")  # preferred raw input: 1 record per page
assert PAGES_JSONL.exists(), f"File not found: {PAGES_JSONL.resolve()}"

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file (one JSON object per line) into a list of dicts."""
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

pages = load_jsonl(PAGES_JSONL)
print("Loaded pages:", len(pages))
print("Example keys:", pages[0].keys())
print("Example page_id:", pages[0].get("page_id"))
print("Example text sample:\n", pages[0]["text"][:400])


Fixed-size chunking means: 
- Split text by character count (proxy for tokens)
- Simplest baseline to compare against more advanced methods

In [109]:
def fixed_size_chunks(
    text: str,
    chunk_chars: int = 1600,
    min_chars: int = 200
) -> List[str]:
    """
    Split text into fixed-size chunks by character length.

    For Spanish, ~1600 chars is often ~300–500 tokens (roughly).
    """
    text = text.strip()
    if not text:
        print("[ERR] empty text")
        return []

    out = []
    for start in range(0, len(text), chunk_chars):
        chunk = text[start:start + chunk_chars].strip()
        if len(chunk) >= min_chars:
            out.append(chunk)
    return out

In [114]:
# Demo on one page
demo = pages[0]["text"]
demo_chunks = fixed_size_chunks(demo, chunk_chars=200, min_chars=10)
print("Chunks:", len(demo_chunks))
print("First chunk sample:\n", demo_chunks[0][:])
print("Second chunk sample:\n", demo_chunks[1])

Chunks: 11
First chunk sample:
 EL MANUAL PARA EL PACIENTE ONCOLÓGICO Y SU FAMILIA es fruto de una intensa reflexión y de una gran empatía con el paciente oncológico. Sin intentar rebajar un ápice la importancia que tiene la enferme
Second chunk sample:
 dad ni negar su dureza, los autores han querido obviar la visión catastrofista, tan habitual cuando se habla del cáncer y han preferido incidir en los aspectos que más pueden servir a los pacientes pa


#### Chunking with overlap

Overlap helps because: 
- Answers often span fixed-size boundaries
- reduces the chance of cutting chunk context in half

In [115]:
def fixed_size_chunks_with_overlap(
    text: str,
    chunk_chars: int = 1600,
    overlap_chars: int = 200,
    min_chars: int = 200
) -> List[str]:
    """
    Sliding window chunking: fixed size + overlap.

    Important: overlap_chars must be < chunk_chars.
    We also ensure the step makes progress (avoids infinite loops).
    """
    text = text.strip()
    if not text:
        print("[ERR] empty text")
        return []
    if overlap_chars >= chunk_chars:
        raise ValueError("overlap_chars must be < chunk_chars")

    out = []
    start = 0
    step = chunk_chars - overlap_chars

    while start < len(text):
        end = min(start + chunk_chars, len(text))
        chunk = text[start:end].strip()
        if len(chunk) >= min_chars:
            out.append(chunk)
        if end == len(text):
            break
        start += step

    return out


In [117]:
demo_chunks2 = fixed_size_chunks_with_overlap(demo, chunk_chars=200, overlap_chars=50, min_chars=50)
print("Chunks:", len(demo_chunks2))
print("Chunk0 tail:\n", demo_chunks2[0])
print("Chunk1 head:\n", demo_chunks2[1])  # should overlap with chunk0 tail

Chunks: 14
Chunk0 tail:
 EL MANUAL PARA EL PACIENTE ONCOLÓGICO Y SU FAMILIA es fruto de una intensa reflexión y de una gran empatía con el paciente oncológico. Sin intentar rebajar un ápice la importancia que tiene la enferme
Chunk1 head:
 bajar un ápice la importancia que tiene la enfermedad ni negar su dureza, los autores han querido obviar la visión catastrofista, tan habitual cuando se habla del cáncer y han preferido incidir en los


<a id='3'></a>
## Variable-size chunking

<a id='3-1'></a>
### Recursive Character Splitting 

Tries "nice" boundaries first: 
- paragraph breaks
- sentences
- spaces
- characters (last resort)

Which usually produces more coherent chunks than raw fixed-size. 
You can tune chunk_chars / min_chars to match your embedding model.

The following method mixes fixed and variable sized chunking while performing first a structure-aware chunking (paragraph/sentece) and then enforce a max chunk size with a final fixed split if needed. 

In [ ]:
#Splits text on blank lines (a newline, optional whitespace)
def split_into_paragraphs(text: str) -> List[str]:
    """Split on blank lines; good when paragraphs are preserved."""
    parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    return parts

#Split after ., ? or ! and keeps the punctuation attached to the sentence
def split_into_sentences_simple(text: str) -> List[str]:
    """
    Simple sentence splitter (heuristic).
    Not perfect, but works decently for Spanish in many cases.
    """
    sents = re.split(r"(?<=[\.\?\!])\s+", text.strip())
    return [s.strip() for s in sents if s.strip()]

def recursive_character_split(
    text: str,
    chunk_chars: int = 1600,
    min_chars: int = 200
) -> List[str]:
    """
    Variable-size chunker:
    - Try to build chunks by paragraphs first
    - If paragraphs are too big, fall back to sentences
    - If still too big, fall back to fixed char splitting

    Why this approach?
    - Maintains semantic coherence where possible (paragraph/sentence)
    - Still guarantees chunks won't exceed the desired size

    Goal:
    Produce chunks that:
    - are not longer than chunk_chars
    - are not too small (min_chars)
    - preserve meaning (paragraphs and sentences stay together when possible)
    """
    text = text.strip()
    if not text:
        print("[ERR] empty text")
        return []

    paragraphs = split_into_paragraphs(text)
    # If we have no real paragraphs, fallback to sentences
    if len(paragraphs) <= 1:
        paragraphs = split_into_sentences_simple(text)

    chunks = [] #final output
    cur = "" #the current chunk being built

    #Takes whatever text is currently accumulated in cur, cleans it and if it's big enough, stores it. 
    # Then resets cur to start a nuew chunk
    def flush():
        nonlocal cur #without nonlocal, flush() would think cur is a local variable inside it
        cur = cur.strip()
        if len(cur) >= min_chars:
            chunks.append(cur)
        cur = ""


    for unit in paragraphs: #Iterate over paragraphs or sentences
        unit = unit.strip()
        if not unit: #skip empty ones
            continue

        #CASE 1: Unit is too large on its own
        #Fixed size chunk
        # If a single unit is bigger than the chunk size, split it further
        if len(unit) > chunk_chars:
            # flush current chunk first
            if cur:
                flush()
            # split oversized unit using fixed-size chunking
            for sub in fixed_size_chunks(unit, chunk_chars=chunk_chars, min_chars=min_chars):
                chunks.append(sub)
            continue

        #CASE 2: Unit fits into current chunk
        # Add unit to current chunk if it fits, otherwise flush and start new
        if len(cur) + len(unit) + 2 <= chunk_chars:
            cur = (cur + "\n\n" + unit) if cur else unit
        #CASE 3: Unit doesn't fit, then flush and start new chunk with current unit
        else:
            flush()
            cur = unit

    if cur:
        flush()

    return chunks


In [120]:
demo_chunks3 = recursive_character_split(demo, chunk_chars=800)
print("Chunks:", len(demo_chunks3))
print("Chunk 1:\n",demo_chunks3[0])
print("Chunk 2:\n",demo_chunks3[1])
print("Chunk 3:\n",demo_chunks3[2])

Chunks: 3
Chunk 1:
 EL MANUAL PARA EL PACIENTE ONCOLÓGICO Y SU FAMILIA es fruto de una intensa reflexión y de una gran empatía con el paciente oncológico.

Sin intentar rebajar un ápice la importancia que tiene la enfermedad ni negar su dureza, los autores han querido obviar la visión catastrofista, tan habitual cuando se habla del cáncer y han preferido incidir en los aspectos que más pueden servir a los pacientes para que afronten con éxito su situación, hacerles la vida más agradable durante el tratamiento y ayudarles en su retorno a la vida diaria una vez curados.

Eminentemente práctico, el libro ofrece información rigurosa a la vez que útil, transmitiendo a la par, un mensaje de ánimo y de confianza.
Chunk 2:
 En definitiva, ofrece ideas prácticas y positivas para mejorar la calidad de vida del enfermo oncológico, de sus familiares y de los cuidadores que le atienden.

Mª Luisa de Cáceres Zurita, de quién surgió la idea original, es Doctora en Psicología y especialista en Psicolo

<a id='4'></a>
## Semantic chunking

Semantic chunking splits where meaning shifts, not where characters end. 
A simple approach could be: 
1. Split into sentences
2. Embed each sentence
3. Compute similarity between consecutive sentences
4. Put a boundary when similarity drops below a threshold

NOTES: 
- sim_threshold is a hyperparameter you'll have to tune
- semantic chunk is slower but can improve coherence.

In [121]:
# Load embedding model (you can swap this later)
EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb_model = SentenceTransformer(EMB_MODEL_NAME)

def semantic_chunking(
    text: str,
    model: SentenceTransformer,
    max_chunk_chars: int = 1600,
    min_chars: int = 200,
    sim_threshold: float = 0.55
) -> List[str]:
    """
    Semantic chunking via sentence embeddings:
    - Split into sentences
    - If there is only one sentence, apply fixed-size chunking
    - Embed each sentence
    - Break chunk when similarity to next sentence drops under threshold
    - Also enforce max_chunk_chars

    Why do this?
    - Helps keep each chunk about "one idea" which often improves retrieval quality
    """
    sentences = split_into_sentences_simple(text)
    if not sentences:
        return []
    if len(sentences) == 1:
        return fixed_size_chunks(sentences[0], chunk_chars=max_chunk_chars, min_chars=min_chars)

    sent_vecs = model.encode(sentences, normalize_embeddings=True)
    chunks = []
    cur_sents = [sentences[0]]
    cur_len = len(sentences[0])

    for i in range(len(sentences) - 1):
        # cosine similarity because normalized embeddings
        sim = float(np.dot(sent_vecs[i], sent_vecs[i+1]))

        nxt = sentences[i+1]
        # boundary if meaning shifts OR chunk too big
        if sim < sim_threshold or (cur_len + 1 + len(nxt) > max_chunk_chars):
            chunk = " ".join(cur_sents).strip()
            if len(chunk) >= min_chars:
                chunks.append(chunk)
            cur_sents = [nxt]
            cur_len = len(nxt)
        else:
            cur_sents.append(nxt)
            cur_len += 1 + len(nxt)

    # flush
    chunk = " ".join(cur_sents).strip()
    if len(chunk) >= min_chars:
        chunks.append(chunk)

    return chunks


In [124]:
demo_sem = semantic_chunking(demo, emb_model, max_chunk_chars=800, sim_threshold=0.55)
print("Semantic chunks:", len(demo_sem))
print("Chunk 1:\n",demo_sem[0])
print("Chunk 2:\n",demo_sem[1])

Semantic chunks: 2
Chunk 1:
 Sin intentar rebajar un ápice la importancia que tiene la enfermedad ni negar su dureza, los autores han querido obviar la visión catastrofista, tan habitual cuando se habla del cáncer y han preferido incidir en los aspectos que más pueden servir a los pacientes para que afronten con éxito su situación, hacerles la vida más agradable durante el tratamiento y ayudarles en su retorno a la vida diaria una vez curados.
Chunk 2:
 Manual para el paciente oncológico y su familia Mª Luisa de Cáceres Zurita, Psico-oncóloga Francisca Ruiz Mata, Diplomada en Enfermería Jose Rámon Germà Lluch, Médico Oncólogo Cristina Carlota Busques, Diplomada en Trabajo Social Manual para el paciente oncológico y su familia Manual para el paciente oncológico y su familia ESP04/070N4


<a id='5'></a>
## LLM Chunking

LLM chunking often means:

- Ask an LLM to segment text into labeled sections: *“Síntomas”, “Tratamiento”, “Afrontamiento emocional”…*
- Or extract an outline and chunk by outline

You'd call your LLM and ask it to:
- produce section boundaries, OR
- rewrite into structured sections, OR
- return JSON with chunks + titles

TODO: Don't need to evaluate the chunkings. Just select a method and explore the use of a vector database (vertex?)

<a id='6'></a>
## Searching

To compare methods, let's build a consistent retrieval layer. We'll implement a simple local search: 
1. Build chunks with a chosen method
2. Embed all chunks
3. Given a query, embed query
4. Cosine similarity -> top-k results
5. Print results with page citation metadata

<a id='6-1'></a>
### Build a chunk dataset for one method

In [ ]:
def build_chunks_from_pages(
    pages: List[Dict[str, Any]],
    chunker_name: str,
    chunker_fn,
) -> List[Dict[str, Any]]:
    """
    Convert page records into chunk records (embedding units).
    We keep the citation metadata: doc_id, page, source, page_id.
    """
    out = []
    for pg in pages:
        text = pg["text"]
        subtexts = chunker_fn(text)

        for j, sub in enumerate(subtexts):
            out.append({
                "chunk_id": f'{pg["page_id"]}__{chunker_name}__c{j:03d}',
                "text": sub,
                "chunk_index": j,

                # citation / traceability
                "page": pg.get("page"),
                "page_id": pg.get("page_id"),
                "doc_id": pg.get("doc_id"),
                "source": pg.get("source"),
                "topic": pg.get("topic"),
                "lang": pg.get("lang"),
                "path": pg.get("path"),
            })
    return out

<a id='6-2'></a>
### Embed + search utilities

In [ ]:
def embed_texts(model: SentenceTransformer, texts: List[str], batch_size: int = 64) -> np.ndarray:
    """Embed a list of texts and return a (N, D) numpy array."""
    vecs = model.encode(texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    return np.asarray(vecs)

def search_top_k(
    model: SentenceTransformer,
    chunk_vecs: np.ndarray,
    chunks: List[Dict[str, Any]],
    query: str,
    top_k: int = 5
) -> List[Tuple[float, Dict[str, Any]]]:
    """
    Local vector search:
    - embed query
    - dot product with normalized chunk vectors = cosine similarity
    - return top_k chunks + scores
    """
    q = model.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vecs @ q
    idx = np.argsort(-scores)[:top_k]
    return [(float(scores[i]), chunks[i]) for i in idx]

<a id='6-3'></a>
### Run a search demo for one method

In [ ]:
# Choose a chunking method to test quickly
chunker_name = "fixed_overlap"
chunker_fn = lambda t: fixed_size_chunks_with_overlap(t, chunk_chars=1600, overlap_chars=200, min_chars=200)

chunks = build_chunks_from_pages(pages, chunker_name, chunker_fn)
print("Total chunks:", len(chunks))

chunk_vecs = embed_texts(emb_model, [c["text"] for c in chunks], batch_size=64)
print("Vectors shape:", chunk_vecs.shape)

query = "cómo manejar la ansiedad después del diagnóstico"
hits = search_top_k(emb_model, chunk_vecs, chunks, query, top_k=5)

for rank, (score, ch) in enumerate(hits, 1):
    print(f"\n#{rank} score={score:.4f} | {ch['source']} | {ch['doc_id']} | p.{ch['page']} | {ch['chunk_id']}")
    print(ch["text"][:450])

<a id='7'></a>
## Evaluation of chunking methods and metrics

You need two evaluation layers:

A) Cheap, automatic metrics (good for iteration)

- **chunk stats**: chunks per page, avg length
- **retrieval self-consistency**: query retrieves same doc/topic
- **latency** estimates: number of chunks and embedding time

B) Real retrieval metrics (needs a labeled eval set)

To compute Recall@k / MRR properly, you need:

- a set of test queries
- a set of “relevant chunk ids” or “relevant pages/doc_ids” for each query

<a id='7-1'></a>
### Chunk statistics

In [ ]:
def chunk_stats(chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    lengths = [len(c["text"]) for c in chunks]
    by_page = Counter(c["page_id"] for c in chunks)

    return {
        "n_chunks": len(chunks),
        "len_min": int(np.min(lengths)) if lengths else 0,
        "len_avg": float(np.mean(lengths)) if lengths else 0.0,
        "len_max": int(np.max(lengths)) if lengths else 0,
        "chunks_per_page_min": int(min(by_page.values())) if by_page else 0,
        "chunks_per_page_avg": float(np.mean(list(by_page.values()))) if by_page else 0.0,
        "chunks_per_page_max": int(max(by_page.values())) if by_page else 0,
    }

print(chunk_stats(chunks))

<a id='7-2'></a>
### Evaluation dataset

TODO: Create a small list of manually at first (10-30 queries). For each query, label relevant doc_id or page_id. 

In [ ]:
EVAL_SET = [
    {
        "query": "efectos secundarios comunes de la quimioterapia",
        # start simple: relevant doc_ids or page_ids you know contain the answer
        "relevant_doc_ids": ["mama_es_novartis_guia-pacientes-cm_2025"],
        "relevant_page_ids": []  # optionally fill later
    },
    {
        "query": "cómo hablar con la familia sobre el diagnóstico",
        "relevant_doc_ids": [],
        "relevant_page_ids": []
    },
]

<a id='7-3'></a>
### Metrics: Recall@k and MRR (by doc_id)

Treat retrieval as correct if any of the top-k chunks comes from a relevant doc

In [ ]:
def recall_at_k_doc(hits: List[Tuple[float, Dict[str, Any]]], relevant_doc_ids: List[str], k: int) -> float:
    if not relevant_doc_ids:
        return float("nan")
    top = hits[:k]
    got = any(h[1].get("doc_id") in set(relevant_doc_ids) for h in top)
    return 1.0 if got else 0.0

def mrr_doc(hits: List[Tuple[float, Dict[str, Any]]], relevant_doc_ids: List[str]) -> float:
    if not relevant_doc_ids:
        return float("nan")
    rel = set(relevant_doc_ids)
    for rank, (_, ch) in enumerate(hits, 1):
        if ch.get("doc_id") in rel:
            return 1.0 / rank
    return 0.0

def evaluate_chunking_method(
    pages: List[Dict[str, Any]],
    chunker_name: str,
    chunker_fn,
    model: SentenceTransformer,
    eval_set: List[Dict[str, Any]],
    top_k: int = 10
) -> Dict[str, Any]:
    # Build and embed corpus chunks
    chunks = build_chunks_from_pages(pages, chunker_name, chunker_fn)
    vecs = embed_texts(model, [c["text"] for c in chunks], batch_size=64)

    # Run eval
    r5, r10, mrrs = [], [], []
    for ex in eval_set:
        hits = search_top_k(model, vecs, chunks, ex["query"], top_k=top_k)
        r5.append(recall_at_k_doc(hits, ex.get("relevant_doc_ids", []), k=5))
        r10.append(recall_at_k_doc(hits, ex.get("relevant_doc_ids", []), k=10))
        mrrs.append(mrr_doc(hits, ex.get("relevant_doc_ids", [])))

    def nanmean(xs):
        xs = [x for x in xs if not (isinstance(x, float) and math.isnan(x))]
        return float(np.mean(xs)) if xs else float("nan")

    return {
        "chunker": chunker_name,
        **chunk_stats(chunks),
        "recall@5_doc": nanmean(r5),
        "recall@10_doc": nanmean(r10),
        "mrr_doc": nanmean(mrrs),
    }


<a id='7-4'></a>
### Compare multiple chunkers

In [ ]:
chunkers = {
    "fixed_no_overlap": lambda t: fixed_size_chunks(t, chunk_chars=1600, min_chars=200),
    "fixed_overlap": lambda t: fixed_size_chunks_with_overlap(t, chunk_chars=1600, overlap_chars=200, min_chars=200),
    "recursive": lambda t: recursive_character_split(t, chunk_chars=1600, min_chars=200),
    "semantic": lambda t: semantic_chunking(t, emb_model, max_chunk_chars=1600, min_chars=200, sim_threshold=0.55),
}

results = []
for name, fn in chunkers.items():
    print("\nEvaluating:", name)
    res = evaluate_chunking_method(pages, name, fn, emb_model, EVAL_SET, top_k=10)
    results.append(res)

results